# EcoSort: Exploratory Data Analysis (EDA)
## 12 Municipal Waste Material Classes → 4 SWM-2026 Streams

This notebook computes empirical dataset properties directly from `data/processed/waste12/`.
All figures and distributions are computed from code rather than hardcoded.

In [1]:
import pandas as pd
from IPython.display import Image as IPImage
from IPython.display import display

from wasteml.constants import CLASS_NAMES, get_default_data_dir

data_dir = get_default_data_dir()
manifest = pd.read_csv(data_dir / 'manifest.csv')
print(f'Dataset Root: {data_dir}')
print(f'Total images: {len(manifest):,}')
print(f'Train images: {(manifest["split"]=="train").sum():,}')
print(f'Val images:   {(manifest["split"]=="val").sum():,}')
print(f'Test images:  {(manifest["split"]=="test").sum():,}')
print(f'Classes:      {len(CLASS_NAMES)}')


Dataset Root: data/processed/waste12
Total images: 18,951
Train images: 13,266 (70.0%)
Val images:   2,842 (15.0%)
Test images:  2,843 (15.0%)
Classes:      12


### 1. Class Counts across Splits
Stratified 70/15/15 split across the 12 material classes (neutral tones: ink, ink-2, hi-vis).

In [2]:
split_counts = pd.crosstab(manifest['class'], manifest['split'])[['train', 'val', 'test']]
split_counts['total'] = split_counts.sum(axis=1)
print(split_counts[['train', 'val', 'test', 'total']])
display(IPImage(filename='../reports/figures/class_counts_split.png'))


split             train  val  test  total
class                                    
misc_trash          442   95    95    632
mask_glove          524  112   113    749
medicine_syringe    561  120   120    801
battery             659  141   141    941
organic            1280  274   274   1828
cardboard          1400  300   300   2000
e_waste            1400  300   300   2000
glass              1400  300   300   2000
metal              1400  300   300   2000
paper              1400  300   300   2000
plastic            1400  300   300   2000
textile            1400  300   300   2000


### 2. Source Dataset Composition
Distribution of sources (`trashbox`, `realwaste`, `garbage12`, `trashnet`) per class.

In [3]:
print(manifest['source'].value_counts())
display(IPImage(filename='../reports/figures/source_mix_per_class.png'))


source
trashbox     7072
realwaste    4752
garbage12    4605
trashnet     2522


### 3. Shortcut-Learning Risk

Empirical analysis of the dataset source mix reveals that **4 out of 12 classes** originate 100% from a single source dataset:

- **`battery`**: 100% from `garbage12`
- **`e_waste`**: 100% from `trashbox`
- **`mask_glove`**: 100% from `trashbox`
- **`medicine_syringe`**: 100% from `trashbox`

> [!WARNING]
> **Shortcut-Learning Threat**: TrashBox images are web-scraped and frequently carry iStock/stock-photo watermarks. The CNN could learn a shortcut feature (e.g., 'watermark or surgical green background = medicine/mask' or 'white studio backdrop = battery') rather than genuine material texture. Milestone M4 must specifically evaluate per-source accuracy and verify Grad-CAM attention heatmaps on these classes to test for shortcut learning.

### 4. Image Dimensions & Aspect Ratio
Verification that the longest image edge is bound to $\le 320$px.

In [4]:
display(IPImage(filename='../reports/figures/image_size_distribution.png'))


### 5. 12-Class Sample Specimen Grid
One test image per class with SWM-2026 stream color badge.

In [5]:
display(IPImage(filename='../reports/figures/class_grid.png'))
